#### 2026-01-30\_gp\_correction\_2022.ipynb
**Author**: Jay Sayre
**Date Modified**: 2026-01-30
**Description**: Fits a Gaussian Process correction layer on top of deep model yield predictions, following You et al. (2017). Trains on municipality-level SIAP ground truth with a squared-exponential spatial kernel over municipality centroids. Hyperparameters are tuned via leave-one-state-out cross-validation. Outputs GP-corrected ADC-level predictions.

**Inputs**:
- `adc_yield_preds_corrected_2022.csv` — RS-based yield predictions
- `adc_alpha_earth_preds.csv` — AEF standard yield predictions
- `adc_mlp_yield_preds.csv` — AEF MLP yield predictions
- `adc_land_use_ca22_adc07.dta` — INEGI 2022 census data (for planted area weights)
- `siap_ag_prod_estimation_ca2007.dta` — SIAP municipality-level production estimates
- `MUNICIPIOS.shp` — Municipality polygon shapefile for centroids

**Outputs**:
- `adc_gp_yield_preds_2022.csv` — GP-corrected yield predictions at ADC level

In [ ]:
import pandas as pd
import geopandas as gpd
import os
import numpy  as np
from   scipy.spatial.distance import cdist
from   itertools import product

### Directories
dropbox_dir   =  os.path.join(os.path.expanduser("~"), "Dropbox", "Projects")
poppy_dir     =  os.path.join(dropbox_dir,  "Maize_prediction")
crop_sub_dir  =  os.path.join(dropbox_dir,  "The Promise of Crop Substitution")
siap_dir      =  os.path.join(crop_sub_dir, "data", "SIAP", "Cleaned")
joel_dir      =  os.path.join(poppy_dir,    "Data", "predictions")
py_md_lab_dir =  os.path.join(poppy_dir,    "Data", "INEGI", "MD_lab_outputs")
ca2022_adc_d  =  os.path.join(py_md_lab_dir,"LM2304-CA22-2025-09-29-superficie_ENTREGA")
mun_shp_dir   =  os.path.join(dropbox_dir,  "Avocado_Deforestation", "Data", "raw", "spatial", "Municipality_shp")

### Inputs
adc_22_07_path =  os.path.join(ca2022_adc_d, "adc_land_use_ca22_adc07.dta")   # census (for area weights)
rs_preds_file  =  os.path.join(joel_dir, "adc_yield_preds_corrected_2022.csv") # RS-based predictions
aef_preds_file =  os.path.join(joel_dir, "adc_alpha_earth_preds.csv")          # AEF standard
mlp_preds_file =  os.path.join(joel_dir, "adc_mlp_yield_preds.csv")            # AEF MLP
siap_path      =  os.path.join(siap_dir, "siap_ag_prod_estimation_ca2007.dta") # SIAP municipal yields
mun_shp_path   =  os.path.join(mun_shp_dir, "MUNICIPIOS.shp")                 # municipality polygons

### Outputs
gp_out_path    =  os.path.join(joel_dir, "adc_gp_yield_preds_2022.csv")        # GP-corrected predictions

#### Load Data

In [ ]:
### ------------------------------------------------------------------ ###
### Load ADC census data (for planted area weights)
### ------------------------------------------------------------------ ###

adc_df             =  pd.read_stata(adc_22_07_path)
adc_df             =  adc_df[adc_df['name'] == 'Maize']
adc_df             =  adc_df[['adc', 'muncode', 'land_input']]

### ------------------------------------------------------------------ ###
### Load prediction files
### ------------------------------------------------------------------ ###

rs_df              =  pd.read_csv(rs_preds_file)
rs_df['adc']       =  rs_df['adcid'].str.replace('-', '', regex=False)
rs_df              =  rs_df.rename(columns={'yield_pred_ls': 'yield_pred_rs'})
rs_df              =  rs_df[['adc', 'yield_pred_rs']]

aef_df             =  pd.read_csv(aef_preds_file)
aef_df['adc']      =  aef_df['adcid'].str.replace('-', '', regex=False)
aef_df             =  aef_df.rename(columns={'yield_pred': 'yield_pred_aef'})
aef_df             =  aef_df[['adc', 'yield_pred_aef']]

mlp_df             =  pd.read_csv(mlp_preds_file)
mlp_df             =  mlp_df[mlp_df['year'] == 2022]
mlp_df['adc']      =  mlp_df['adcid'].str.replace('-', '', regex=False)
mlp_df             =  mlp_df.rename(columns={'pred_yield': 'yield_pred_mlp'})
mlp_df             =  mlp_df[['adc', 'yield_pred_mlp']]

### ------------------------------------------------------------------ ###
### Load SIAP municipal yields (2022, Maize)
### ------------------------------------------------------------------ ###

siap_df            =  pd.read_stata(siap_path)
siap_df            =  siap_df[siap_df['name'] == 'Maize']
siap_df['yield']   =  siap_df['q'] / siap_df['ha_planted']
siap_df['muncode'] =  siap_df['muncode'].apply(lambda x: str(int(x)).zfill(5))
siap_df            =  siap_df[siap_df['year'] == 2022]
siap_df            =  siap_df[['muncode', 'yield']].rename(columns={'yield': 'yield_siap'})

### ------------------------------------------------------------------ ###
### Load municipality centroids
### ------------------------------------------------------------------ ###

mun_gdf             =  gpd.read_file(mun_shp_path)
mun_gdf['muncode']  =  mun_gdf['CVE_ENT'] + mun_gdf['CVE_MUN']
mun_gdf             =  mun_gdf.to_crs(epsg=4326)
mun_gdf['cen_lon']  =  mun_gdf.geometry.centroid.x
mun_gdf['cen_lat']  =  mun_gdf.geometry.centroid.y
mun_centroids       =  mun_gdf[['muncode', 'cen_lat', 'cen_lon']].copy()

### ------------------------------------------------------------------ ###
### Merge into single ADC-level DataFrame
### ------------------------------------------------------------------ ###

df = adc_df.copy()
df = df.merge(rs_df,  on='adc', how='left')
df = df.merge(aef_df, on='adc', how='left')
df = df.merge(mlp_df, on='adc', how='left')
df = df.merge(mun_centroids, on='muncode', how='left')

df['CVE_ENT'] = df['muncode'].str.slice(0, 2)

print(f"ADCs loaded:       {len(df):,}")
print(f"  RS predictions:  {df['yield_pred_rs'].notna().sum():,}")
print(f"  AEF predictions: {df['yield_pred_aef'].notna().sum():,}")
print(f"  MLP predictions: {df['yield_pred_mlp'].notna().sum():,}")
print(f"  Centroids:       {df['cen_lat'].notna().sum():,}")

#### Gaussian Process Correction (You et al. 2017)

Semi-parametric correction following You et al. (2017, eq. 1--3):

$$y(x) = f(x) + h(x)^\top \beta$$

where $f(x) \sim \mathcal{GP}(0, k)$ captures spatially-correlated residual structure, $h(x) = [\hat{y}_{\text{model}},\; 1]$ are basis functions (scalar model prediction + intercept), and $\beta \sim \mathcal{N}(b, B)$ with prior $b = [1, 0]$, $B = \sigma_b I$.

The kernel is squared-exponential over municipality centroid coordinates (lat/lon):

$$k(x, x') = \sigma^2 \exp\!\left(-\frac{\|g - g'\|^2}{2\, r_{\text{loc}}^2}\right) + \sigma_e^2 \, \delta_{x,x'}$$

Hyperparameters $(\sigma, \sigma_b, \sigma_e, r_{\text{loc}})$ are tuned via leave-one-state-out cross-validation on municipality-level SIAP ground truth. Final GP is fit on all training municipalities and used to predict at the ADC level.

In [ ]:
### ------------------------------------------------------------------ ###
### GP correction: implementation (You et al. 2017 / Rasmussen 2006 §2.7)
### ------------------------------------------------------------------ ###

def deep_gp_predict(H_train, y_train, D2_train, H_test, D2_test_train,
                    sigma, sigma_b, sigma_e, r_loc, b_prior):
    """GP prediction with basis functions (Rasmussen 2006, eq 2.41).

    Parameters
    ----------
    H_train       : (N_train, p)  basis function matrix [pred, 1]
    y_train       : (N_train,)    SIAP yields
    D2_train      : (N_train, N_train) precomputed squared-distance matrix
    H_test        : (N_test, p)   basis functions at test points
    D2_test_train : (N_test, N_train) precomputed squared distances test→train
    sigma, sigma_b, sigma_e, r_loc : GP hyperparameters
    b_prior       : (p,) prior mean for beta

    Returns
    -------
    preds : (N_test,) predicted yields
    """
    N =  len(y_train)
    p =  H_train.shape[1]

    ### Prior precision for beta
    B_inv =  np.eye(p) / (sigma_b**2)

    ### Build training kernel from precomputed D2
    K  =  sigma**2 * np.exp(-D2_train / (2.0 * r_loc**2))
    K +=  sigma_e**2 * np.eye(N)

    ### Cholesky for numerical stability
    try:
        L =  np.linalg.cholesky(K)
    except np.linalg.LinAlgError:
        K +=  1e-6 * np.eye(N)
        L  =  np.linalg.cholesky(K)

    ### Solve via Cholesky (avoid explicit inverse)
    ### K_inv @ M  =  cho_solve(L, M)
    K_inv_H =  np.linalg.solve(L, H_train)
    K_inv_H =  np.linalg.solve(L.T, K_inv_H)      # (N, p)
    K_inv_y =  np.linalg.solve(L, y_train)
    K_inv_y =  np.linalg.solve(L.T, K_inv_y)       # (N,)

    ### Posterior mean of beta
    HtKinv_H =  H_train.T @ K_inv_H                # (p, p)
    A        =  B_inv + HtKinv_H                    # (p, p)
    rhs      =  K_inv_H.T @ y_train + B_inv @ b_prior
    beta_bar =  np.linalg.solve(A, rhs)             # (p,)

    ### Residual and alpha
    resid =  y_train - H_train @ beta_bar
    alpha =  np.linalg.solve(L, resid)
    alpha =  np.linalg.solve(L.T, alpha)

    ### Vectorized prediction at all test points
    K_star =  sigma**2 * np.exp(-D2_test_train / (2.0 * r_loc**2))  # (N_test, N_train)
    preds  =  H_test @ beta_bar + K_star @ alpha

    return preds


def gp_loso_cv_fast(H_all, y_all, D2_full, state_ids, states,
                    sigma, sigma_b, sigma_e, r_loc, b_prior):
    """Fast leave-one-state-out CV using precomputed arrays.
    Returns RMSE across all held-out municipalities."""
    all_resid =  []

    for st in states:
        mask_te =  state_ids == st
        mask_tr =  ~mask_te
        n_tr    =  mask_tr.sum()
        n_te    =  mask_te.sum()

        if n_tr < 10 or n_te == 0:
            continue

        preds =  deep_gp_predict(
            H_all[mask_tr],  y_all[mask_tr],
            D2_full[np.ix_(mask_tr, mask_tr)],
            H_all[mask_te],
            D2_full[np.ix_(mask_te, mask_tr)],
            sigma, sigma_b, sigma_e, r_loc, b_prior
        )
        all_resid.append(y_all[mask_te] - preds)

    if len(all_resid) == 0:
        return np.inf
    all_resid =  np.concatenate(all_resid)
    return np.sqrt(np.mean(all_resid**2))

#### Tune Hyperparameters and Predict

In [ ]:
### ------------------------------------------------------------------ ###
### GP correction: build mun-level training data, tune, and predict
### ------------------------------------------------------------------ ###

raw_pred_cols  =  ['yield_pred_rs', 'yield_pred_aef', 'yield_pred_mlp']
b_prior        =  np.array([1.0, 0.0])  # prior: raw prediction is a priori correct

### Hyperparameter grid
sigma_grid     =  [0.5, 1.0, 2.0, 4.0]
sigma_b_grid   =  [0.1, 0.5, 1.0, 5.0]
sigma_e_grid   =  [0.1, 0.5, 1.0, 2.0]
r_loc_grid     =  [0.5, 1.0, 2.0, 5.0]

### --- Build municipality-level area-weighted predictions --- ###
for pc in raw_pred_cols:
    df['_wQ_' + pc] =  df[pc] * df['land_input']
    df['_wA_' + pc] =  df.apply(
        lambda x, col=pc: x['land_input'] if np.isfinite(x[col]) else 0, axis=1
    )

agg_tmp  =  ['_wQ_' + pc for pc in raw_pred_cols] + ['_wA_' + pc for pc in raw_pred_cols]
mun_gp   =  df.groupby('muncode')[agg_tmp].sum().reset_index()

for pc in raw_pred_cols:
    mun_gp[pc + '_mun'] =  mun_gp['_wQ_' + pc] / mun_gp['_wA_' + pc]
    mun_gp.loc[mun_gp['_wA_' + pc] == 0, pc + '_mun'] = np.nan

mun_gp = mun_gp[['muncode'] + [pc + '_mun' for pc in raw_pred_cols]]
mun_gp = mun_gp.merge(siap_df, on='muncode', how='left')
mun_gp = mun_gp.merge(mun_centroids, on='muncode', how='left')
mun_gp['CVE_ENT'] = mun_gp['muncode'].str.slice(0, 2)

### --- Tune and predict for each model --- ###
gp_best_params =  {}

for pc in raw_pred_cols:
    tag   =  pc.replace('yield_pred_', '')
    pcmun =  pc + '_mun'

    ### Training data: municipalities with both prediction and SIAP
    mun_tr = mun_gp[[pcmun, 'yield_siap', 'cen_lat', 'cen_lon', 'CVE_ENT']].replace(
        [np.inf, -np.inf], np.nan).dropna().copy()
    print(f"\n{'='*70}")
    print(f"  GP tuning for {tag.upper()} ({len(mun_tr):,} municipalities)")
    print(f"{'='*70}")

    ### Precompute arrays for fast CV
    H_arr     =  np.column_stack([mun_tr[pcmun].values, np.ones(len(mun_tr))])
    y_arr     =  mun_tr['yield_siap'].values
    G_arr     =  mun_tr[['cen_lat', 'cen_lon']].values
    D2_full   =  cdist(G_arr, G_arr, metric='sqeuclidean')
    state_ids =  mun_tr['CVE_ENT'].values
    states    =  sorted(np.unique(state_ids))

    ### Grid search via LOSO CV
    best_rmse   =  np.inf
    best_params =  None
    n_tested    =  0
    n_total     =  len(sigma_grid) * len(sigma_b_grid) * len(sigma_e_grid) * len(r_loc_grid)

    for sig, sb, se, rl in product(sigma_grid, sigma_b_grid, sigma_e_grid, r_loc_grid):
        rmse_cv =  gp_loso_cv_fast(H_arr, y_arr, D2_full, state_ids, states,
                                    sig, sb, se, rl, b_prior)
        n_tested += 1
        if rmse_cv < best_rmse:
            best_rmse   =  rmse_cv
            best_params =  (sig, sb, se, rl)
        if n_tested % 64 == 0:
            print(f"    {n_tested}/{n_total} combos tested, best RMSE so far = {best_rmse:.4f}")

    sig_best, sb_best, se_best, rl_best = best_params
    gp_best_params[tag] = best_params
    print(f"  Best LOSO RMSE = {best_rmse:.4f}")
    print(f"  sigma={sig_best}, sigma_b={sb_best}, sigma_e={se_best}, r_loc={rl_best}")

    ### Fit final GP on all training municipalities, predict at ADC level
    adc_mask =  df[pc].notna() & df['cen_lat'].notna()
    adc_sub  =  df.loc[adc_mask].copy()

    H_adc       =  np.column_stack([adc_sub[pc].values, np.ones(len(adc_sub))])
    G_adc       =  adc_sub[['cen_lat', 'cen_lon']].values
    D2_adc_tr   =  cdist(G_adc, G_arr, metric='sqeuclidean')

    gp_preds =  deep_gp_predict(H_arr, y_arr, D2_full, H_adc, D2_adc_tr,
                                 sig_best, sb_best, se_best, rl_best, b_prior)

    col_gp       =  f'yield_pred_{tag}_gp'
    df[col_gp]   =  np.nan
    df.loc[adc_mask, col_gp] = gp_preds

    n_gp = df[col_gp].notna().sum()
    print(f"  {col_gp}: {n_gp:,} ADC predictions")

#### Save Results

In [ ]:
### ------------------------------------------------------------------ ###
### Save GP-corrected predictions to CSV
### ------------------------------------------------------------------ ###

gp_cols  =  ['adc'] + [f'yield_pred_{pc.replace("yield_pred_", "")}_gp'
                        for pc in raw_pred_cols]
gp_out   =  df[gp_cols].copy()

gp_out.to_csv(gp_out_path, index=False)

print(f"Written: {gp_out_path}")
print(f"  Rows: {len(gp_out):,}")
for c in gp_cols[1:]:
    print(f"  {c}: {gp_out[c].notna().sum():,} non-null")

### Print best hyperparameters for reference
print(f"\nBest hyperparameters:")
for tag, (sig, sb, se, rl) in gp_best_params.items():
    print(f"  {tag:4s}: sigma={sig}, sigma_b={sb}, sigma_e={se}, r_loc={rl}")